# Tema 4. XGBoost

**Técnicas Avanzadas de Predicción en Negocios · Cuaderno de práctica (nivel medio)**

Datos: Encuesta Cuatrienal de Estructura Salarial 2022 (INE), microdatos.

## Qué vamos a hacer

Predecir el salario de trabajadores incluidos en la EES. Ajustamos XGBoost, el boosting de árboles de la última sesión de teoría, y lo comparamos con el random forest con el mismo protocolo: RMSE de validación cruzada en el entrenamiento, con las particiones `cv5`.

0. **Preparación.** Datos, predictores, preprocesador, particiones, regresión lineal y random forest de referencia.
1. **XGBoost en un `Pipeline`.** XGBoost con sus valores por defecto y búsqueda de la tasa de aprendizaje y la profundidad con `GridSearchCV`.
2. **Penalización de las hojas.** El hiperparámetro `reg_lambda` y la regla del error típico.

Ampliación opcional: la parada temprana, que elige el número de árboles, y otras dos implementaciones del boosting, `HistGradientBoostingRegressor` de scikit-learn y `LightGBM`.

- Hay un ejercicio de código y tres «Para pensar» con la respuesta plegada.
- La ampliación añade un ejercicio y dos «Para pensar».
- En cada ejercicio, los comentarios `# TODO` dicen qué hacer.

**Solo si trabajas en Google Colab.** La celda siguiente instala la versión de scikit-learn del entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

## 0. Preparación

Usa el entorno `tap` del curso.

**Núcleos del procesador.**

- Un núcleo es una parte del procesador que realiza cálculos; cuando hay varios, pueden trabajar a la vez.
- Para aprovechar esta capacidad, consultamos con `os.cpu_count()` cuántas unidades de procesamiento reconoce el sistema.
- En el siguiente bloque de código guardamos ese número menos uno en `N_NUCLEOS`, para dejar margen a otras tareas del ordenador.
- Más adelante, al ajustar los modelos, usaremos esta variable como valor del argumento `n_jobs`, que controla cuántas tareas se ejecutan en paralelo. Así podremos reducir el tiempo de cálculo repartiendo tareas independientes, como el entrenamiento de los árboles de un random forest o los ajustes de `GridSearchCV`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Random forest y XGBoost (con la interfaz de scikit-learn).
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Regresión lineal y preprocesado de los predictores.
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# División, validación cruzada y búsqueda de hiperparámetros.
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV

SEMILLA = 42

# Núcleos para repartir el trabajo: todos los del ordenador menos uno (al menos uno).
N_NUCLEOS = max(1, os.cpu_count() - 1)
print("Núcleos para los cálculos en paralelo:", N_NUCLEOS)

### Datos, predictores y división

- Cargamos los microdatos de la EES 2022 y nos quedamos con los trabajadores que tienen un salario positivo y no superior al percentil 99.
- De ellos elegimos al azar 10 000, con la semilla 42.
- Usamos 16 predictores, 14 categóricos y 2 numéricos.
- Dividimos los datos en entrenamiento (80 %) y test (20 %).

In [ ]:
# Datos de la EES 2022 en la web del curso (separador: tabulación).
URL_DATOS = "https://juandmontoro.github.io/tap/data/EES_2022.tab"
df = pd.read_csv(URL_DATOS, sep="\t", low_memory=False)
df.columns = df.columns.str.replace('"', "").str.strip()

# Trabajadores con salario positivo y no superior al percentil 99.
tope = df["RETRINOIN"].quantile(0.99)
salario_admitido = (df["RETRINOIN"] > 0) & (df["RETRINOIN"] <= tope)
limpio = df[salario_admitido]

# 10 000 trabajadores elegidos al azar.
datos = limpio.sample(n=10000, random_state=SEMILLA).reset_index(drop=True)
print("Observaciones:", len(datos))

In [ ]:
PRED_CATEGORICOS = [
    "SEXO", "ESTU", "ANOS2", "TIPOCON", "TIPOJOR", "CNO1", "CNACE",
    "CONTROL", "MERCADO", "REGULACION", "RESPONSA", "NUTS1", "TIPOPAIS", "ESTRATO2",
]
PRED_NUMERICOS = ["ANOANTI", "JAP"]

# X contiene los 16 predictores: categóricos como categorías y numéricos como decimales (float).
X = datos[PRED_CATEGORICOS + PRED_NUMERICOS].copy()
X[PRED_CATEGORICOS] = X[PRED_CATEGORICOS].astype("category")
X[PRED_NUMERICOS] = X[PRED_NUMERICOS].astype(float)

# y es el salario.
y = datos["RETRINOIN"]

# División: 80 % entrenamiento y 20 % test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)
print("Entrenamiento:", len(X_train), "| test:", len(X_test))

### Preprocesador, particiones y modelos de referencia

- Usamos un preprocesador que aplica transformaciones distintas a predictores categóricos y a numéricos. En el caso de los modelos de árboles, también XGBoost, el argumento `escalar=False` evita el escalado (por innecesario) de los predictores numéricos.
- Las particiones `cv5` son las mismas en todo el cuaderno.
- Como referencias calculamos el RMSE de validación cruzada de la regresión lineal (el modelo base) y añadimos el random forest de 100 árboles con `max_features=0.33`. Para acelerar el proceso, en el random forest se reparte el ajuste de los árboles entre `N_NUCLEOS` núcleos con `n_jobs`, sin que cambie el resultado.
- `cross_val_score` ajusta el modelo cinco veces, una por partición de `cv5`, y devuelve sus cinco puntuaciones (score) de validación. Fijando `scoring="neg_root_mean_squared_error"`, devuelve los cinco RMSE con el signo cambiado.

In [ ]:
# Devuelve un preprocesador nuevo, sin ajustar, en cada llamada.
def crear_preprocesador(escalar=True):
    # Numéricas: escaladas, o sin cambios con "passthrough".
    if escalar:
        numericas = StandardScaler()
    else:
        numericas = "passthrough"
    # Categóricas: una columna por categoría (las de menos del 1 % de los trabajadores se agrupan en una).
    return ColumnTransformer([
        ("numericas", numericas, PRED_NUMERICOS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01,
                                      sparse_output=False), PRED_CATEGORICOS),
    ])


# Cinco particiones del entrenamiento, barajadas con la semilla.
cv5 = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

In [ ]:
# Modelo base: preprocesador con escalado y regresión lineal.
lineal = Pipeline([
    ("preprocesado", crear_preprocesador()),
    ("modelo", LinearRegression()),
])

puntuaciones_lineal = cross_val_score(lineal, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)

# Random forest: 100 árboles, el 33 % de las columnas en cada corte.
bosque = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", RandomForestRegressor(n_estimators=100, max_features=0.33,
                                     n_jobs=N_NUCLEOS, random_state=SEMILLA)),
])

puntuaciones_bosque = cross_val_score(bosque, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5)

# scikit-learn devuelve el RMSE con el signo cambiado: lo pasamos a positivo.
print("Regresión lineal | RMSE de validación cruzada:", round(-puntuaciones_lineal.mean()), "euros | error típico:", round(puntuaciones_lineal.std() / np.sqrt(5)))
print("Random forest 0,33 | RMSE de validación cruzada:", round(-puntuaciones_bosque.mean()), "euros | error típico:", round(puntuaciones_bosque.std() / np.sqrt(5)))

**Lectura de los resultados.**

- La regresión lineal tiene un RMSE de validación cruzada de 11 956 euros y el random forest, de unos 11 200 euros.
- Veamos si XGBoost mejora al random forest. Aunque no lo mejorara, sería una alternativa sólida si su error fuera parecido y se ajustara en menos tiempo.

> **Cifras en otro ordenador.** El random forest y XGBoost pueden dar en tu ordenador RMSE que se separan unas decenas de euros de los del texto y, en la ampliación, un número de árboles algo distinto en la parada temprana. Por eso el texto da sus cifras redondeadas. Las comparaciones entre modelos se mantienen.

---

# Bloque 1. XGBoost en un `Pipeline`

## 1.1 XGBoost con sus valores por defecto

XGBoost ajusta los árboles uno detrás de otro.

- Cada nuevo árbol intenta corregir los errores de las predicciones acumuladas hasta ese momento.
- Su aportación se multiplica por una **tasa de aprendizaje** ν (`learning_rate`) antes de sumarse a la predicción acumulada. Esta tasa controla cuánto influye cada nuevo árbol.
- Sus valores por defecto son 100 árboles (`n_estimators`), ν = 0,3 y profundidad 6 (`max_depth`).
- Lo incluimos en un `Pipeline` con el preprocesador sin escalar, como el random forest.

Para ver si el modelo sobreajusta, comparamos su RMSE de validación cruzada con su RMSE de entrenamiento.

- El RMSE de entrenamiento es el error que comete el modelo con los mismos trabajadores con los que se ha ajustado.
- `cross_validate` calcula los dos: con `return_train_score=True`, devuelve en cada partición el RMSE de validación y el de entrenamiento. Por eso aquí lo usamos en lugar de `cross_val_score`.

> **Python que usamos aquí**
> - `XGBRegressor(n_estimators=B, learning_rate=tasa, max_depth=profundidad, random_state=semilla)` crea un XGBoost de regresión con la interfaz de scikit-learn ([documentación](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBRegressor)).
> - `cross_validate(modelo, X, y, scoring=métrica, cv=particiones, return_train_score=True)` devuelve un diccionario con las puntuaciones de validación (`"test_score"`) y de entrenamiento (`"train_score"`) de cada partición ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html)).

In [ ]:
# XGBoost con sus valores por defecto, salvo la semilla.
xgb_defecto = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", XGBRegressor(random_state=SEMILLA)),
])

# RMSE de validación y de entrenamiento en cada una de las cinco particiones, repartidas entre los núcleos.
resultados_defecto = cross_validate(xgb_defecto, X_train, y_train, scoring="neg_root_mean_squared_error",
                                    cv=cv5, return_train_score=True, n_jobs=N_NUCLEOS)

print("XGBoost por defecto | RMSE de entrenamiento:", round(-resultados_defecto["train_score"].mean()), "euros")
print("XGBoost por defecto | RMSE de validación cruzada:", round(-resultados_defecto["test_score"].mean()), "euros | error típico:", round(resultados_defecto["test_score"].std() / np.sqrt(5)))

**Lectura de los resultados.**

- Con sus valores por defecto, XGBoost tiene un RMSE de entrenamiento de unos 5 600 euros y un RMSE de validación cruzada de algo más de 11 400 euros, peor que el del random forest.
- La distancia entre los dos errores indica sobreajuste: con ν = 0,3 y árboles de profundidad 6, el modelo sigue también el ruido de los trabajadores con los que se ajusta.
- En el apartado 1.2 elegimos la tasa de aprendizaje y la profundidad con validación cruzada.

---

## 1.2 La tasa y la profundidad, con `GridSearchCV`

Buscamos `learning_rate` entre 0,03, 0,1 y 0,3 y `max_depth` entre 2, 4 y 6, con 300 árboles. Son nueve combinaciones y, como cada una se ajusta en las cinco particiones, 45 ajustes.

Definimos el `Pipeline` de XGBoost con 300 árboles, la malla y la búsqueda, y ajustamos la búsqueda con el entrenamiento.

In [ ]:
# XGBoost con 300 árboles. La tasa y la profundidad las fija la búsqueda.
xgb = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", XGBRegressor(n_estimators=300, random_state=SEMILLA)),
])

# Malla: tres tasas de aprendizaje y tres profundidades.
malla_xgb = {"modelo__learning_rate": [0.03, 0.1, 0.3], "modelo__max_depth": [2, 4, 6]}

# Búsqueda con las particiones cv5. Sus 45 ajustes se reparten entre los núcleos.
# Con return_train_score=True, GridSearchCV guarda en cv_results_ el RMSE de entrenamiento.
busqueda_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=malla_xgb,
    scoring="neg_root_mean_squared_error",
    cv=cv5,
    return_train_score=True,
    n_jobs=N_NUCLEOS,
)
busqueda_xgb.fit(X_train, y_train)

Recuperamos la tasa y la profundidad de menor RMSE de validación cruzada.

In [ ]:
# Tasa y profundidad con menor RMSE de validación cruzada.
tasa = busqueda_xgb.best_params_["modelo__learning_rate"]
profundidad = busqueda_xgb.best_params_["modelo__max_depth"]
print("Tasa:", tasa, "| profundidad:", profundidad)

Tabulamos `cv_results_`: los RMSE de validación y de entrenamiento (los dos en positivo) y el error típico.

In [ ]:
# Tabla de cv_results_: RMSE de validación y de entrenamiento en positivo, y error típico.
resultados_xgb = pd.DataFrame(busqueda_xgb.cv_results_)
tabla_xgb = resultados_xgb[["param_modelo__learning_rate", "param_modelo__max_depth",
                            "mean_test_score", "std_test_score", "mean_train_score"]].copy()

tabla_xgb["mean_test_score"] = -tabla_xgb["mean_test_score"]
tabla_xgb["mean_train_score"] = -tabla_xgb["mean_train_score"]
tabla_xgb["error_tipico"] = tabla_xgb["std_test_score"] / np.sqrt(5)

# Redondeamos solo los RMSE: la tasa tiene decimales.
display(tabla_xgb.round({"mean_test_score": 0, "std_test_score": 0, "mean_train_score": 0, "error_tipico": 0}))

**Lectura de los resultados.**

- El menor RMSE de validación cruzada está en ν = 0,1 y profundidad 4 (unos 11 000 euros), por delante del random forest.
- Para los datos de entrenamiento, el RMSE es mínimo con ν = 0,3 y profundidad 6 (unos 2 500 euros), pero su RMSE de validación sube a unos 11 800 euros.
- La segunda mejor combinación, ν = 0,03 y profundidad 6, queda a unas decenas de euros, menos que un error típico (unos 200 euros).

---

### Para pensar 1

En la tabla, ¿por qué el RMSE de entrenamiento baja siempre al subir la tasa o la profundidad y el de validación no?

<details><summary>Respuesta</summary>

- Con más profundidad, cada árbol tiene más hojas y combina más predictores.
- Con más tasa, cada árbol corrige una parte mayor de los residuos que dejan los anteriores.
- Las dos cosas aumentan la capacidad de ajuste, y el modelo se acerca cada vez más a los salarios con los que se ajusta, también a su ruido, así que el RMSE de entrenamiento baja siempre.
- El ruido de esos trabajadores no se repite en los de validación, y a partir de cierto punto el RMSE de validación vuelve a subir.

</details>

# Bloque 2. Penalización de las hojas

## 2.1 `reg_lambda`

Al ajustar cada árbol, XGBoost suma a la pérdida una penalización que crece con el cuadrado de los valores de sus hojas. El hiperparámetro λ (`reg_lambda`, 1 por defecto) fija cuánto pesa esa penalización. Con la pérdida cuadrática, cada hoja vale la media de los residuos de sus $n_j$ observaciones multiplicada por

$$F_j = \frac{n_j}{n_j + \lambda},$$

un factor entre 0 y 1.

- Con λ = 10, una hoja de 5 observaciones conserva 5/15 ≈ 0,33 de su media, y una de 500, 500/510 ≈ 0,98.
- λ encoge sobre todo las hojas con pocas observaciones, que son las que más siguen el ruido.
- Buscamos λ entre 0, 1, 10, 100 y 1 000, con la tasa, la profundidad y los 300 árboles del apartado 1.2.
- XGBoost tiene otra penalización, `gamma`, un coste por cada hoja. En estos datos no mejora el RMSE de validación cruzada, y no la usamos.

**Por qué en dos búsquedas.**

- Elegimos λ en una segunda búsqueda, con la tasa y la profundidad ya fijadas en el apartado 1.2.
- Es una simplificación, porque los hiperparámetros pueden interactuar: por ejemplo, con hojas más encogidas pueden convenir árboles más profundos.
- Lo riguroso es buscar los tres a la vez, con 3 × 3 × 5 = 45 combinaciones, que son 225 ajustes frente a los 70 de las dos búsquedas.
- Lo hacemos en dos etapas para que la tabla de λ sea corta y para poder aplicar después la regla del error típico a un solo hiperparámetro.
- Con estos datos, la búsqueda conjunta puede elegir otra combinación, pero su RMSE de validación cruzada baja como mucho unas decenas de euros, menos que el error típico.

Como en el apartado 1.2, definimos el `Pipeline`, la malla de λ y la búsqueda, y ajustamos la búsqueda con el entrenamiento.

In [ ]:
# XGBoost con la tasa y la profundidad elegidas y 300 árboles. λ lo fija la búsqueda.
xgb_lambda = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", XGBRegressor(n_estimators=300, learning_rate=tasa, max_depth=profundidad,
                            random_state=SEMILLA)),
])

# Malla: cinco valores de λ.
malla_lambda = {"modelo__reg_lambda": [0, 1, 10, 100, 1000]}

# Búsqueda con las particiones cv5, repartida entre los núcleos.
busqueda_lambda = GridSearchCV(
    estimator=xgb_lambda,
    param_grid=malla_lambda,
    scoring="neg_root_mean_squared_error",
    cv=cv5,
    return_train_score=True,
    n_jobs=N_NUCLEOS,
)
busqueda_lambda.fit(X_train, y_train)

Recuperamos el λ de menor RMSE de validación cruzada.

In [ ]:
# λ con menor RMSE de validación cruzada.
print("λ con menor RMSE de validación cruzada:", busqueda_lambda.best_params_["modelo__reg_lambda"])

Tabulamos `cv_results_`, como en el apartado 1.2.

In [ ]:
# Tabla de cv_results_: RMSE de validación y de entrenamiento en positivo, y error típico.
resultados_lambda = pd.DataFrame(busqueda_lambda.cv_results_)
tabla_lambda = resultados_lambda[["param_modelo__reg_lambda", "mean_test_score", "std_test_score",
                                  "mean_train_score"]].copy()

tabla_lambda["mean_test_score"] = -tabla_lambda["mean_test_score"]
tabla_lambda["mean_train_score"] = -tabla_lambda["mean_train_score"]
tabla_lambda["error_tipico"] = tabla_lambda["std_test_score"] / np.sqrt(5)

tabla_lambda.round(0)

**Lectura de los resultados.**

- El RMSE para los datos de entrenamiento sube con λ, de unos 8 400 euros con λ = 0 a unos 10 900 con λ = 1 000: las hojas encogidas siguen menos a los trabajadores del ajuste.
- Entre λ = 0 y λ = 100, el RMSE de validación cruzada cambia menos que el error típico (unos 190 euros), y con 1 000 sube a casi 11 400 euros.
- Por eso, el λ de menor RMSE de validación cruzada puede cambiar de un ordenador a otro.

---

La **regla del error típico** elige el modelo más sencillo cuyo RMSE de validación cruzada no supera el mínimo más su error típico. Con λ el orden de sencillez está claro:

**λ mayor → hojas más encogidas → modelo más sencillo.**

### Ejercicio 1. La regla del error típico con λ

1. Calcula el límite de la regla (el RMSE mínimo más su error típico) y el mayor λ cuyo RMSE de validación cruzada no lo supera.
2. ¿Coinciden el λ del mínimo y el de la regla? ¿Cuál elegirías?

In [ ]:
# TODO: adapta a la búsqueda de λ estas líneas, que aplican la regla del error típico a un árbol de decisión:
#     rmse_arbol = tabla_arbol.loc[busqueda_arbol.best_index_, "mean_test_score"]
#     et_arbol = tabla_arbol.loc[busqueda_arbol.best_index_, "error_tipico"]
#     limite_arbol = rmse_arbol + et_arbol
#     dentro_arbol = tabla_arbol[tabla_arbol["mean_test_score"] <= limite_arbol]
#     print("Mayor min_samples_leaf dentro del límite:", dentro_arbol["param_modelo__min_samples_leaf"].max())
#   Cambia busqueda_arbol y tabla_arbol por busqueda_lambda y tabla_lambda, la columna del hiperparámetro
#   por "param_modelo__reg_lambda" y los nombres nuevos: rmse_lambda, et_lambda, limite_lambda y dentro_lambda.
#   Imprime también el límite.

Paso 2. ¿Coinciden el λ del mínimo y el de la regla? ¿Cuál elegirías?

*Escribe aquí tu respuesta.*

### Para pensar 2

En un random forest, la regla del error típico no sirve para elegir `max_features`, la fracción de columnas que el random forest elige al azar en cada corte. ¿Por qué con λ sí se puede aplicar?

<details><summary>Respuesta</summary>

- La regla necesita saber qué valor del hiperparámetro da un modelo más sencillo.
- Con λ está claro: un λ mayor encoge más las hojas, y el RMSE de entrenamiento lo muestra, porque sube con λ.
- Con `max_features`, cualquier fracción da árboles sin límite del mismo tamaño, así que no hay un orden de sencillez entre sus valores.
- Con el número de árboles del boosting sí lo hay: menos árboles dan un modelo más sencillo.

</details>

# Cierre. XGBoost frente al random forest

Para comparar los modelos del cuaderno, reunimos sus valores de RMSE en validación cruzada.

- Estos valores se calculan con `cross_val_score`, utilizando en todos los casos las mismas particiones, definidas en `cv5`.
- En el caso de XGBoost, incluimos el modelo seleccionado por cada búsqueda de hiperparámetros. Lo recuperamos mediante el atributo `best_estimator_`: es el `Pipeline` con la combinación de hiperparámetros que obtuvo el menor RMSE medio en validación cruzada.
- Para añadir estos modelos a la comparación, calculamos su RMSE con `cross_val_score`, como el de los demás.
- Como las particiones son las mismas, su RMSE medio y su error típico coinciden con los de su fila en `tabla_xgb` o en `tabla_lambda`, según la búsqueda de la que procedan.

In [ ]:
# Los cinco RMSE de los XGBoost de menor RMSE de validación cruzada de las dos búsquedas, repartidos entre los núcleos.
puntuaciones_xgb = cross_val_score(busqueda_xgb.best_estimator_, X_train, y_train,
                                   scoring="neg_root_mean_squared_error", cv=cv5, n_jobs=N_NUCLEOS)
puntuaciones_lambda = cross_val_score(busqueda_lambda.best_estimator_, X_train, y_train,
                                      scoring="neg_root_mean_squared_error", cv=cv5, n_jobs=N_NUCLEOS)

# Resumen de los modelos: RMSE de validación cruzada y error típico de los cinco RMSE de cada modelo.
resumen = pd.DataFrame({
    "modelo": ["regresión lineal", "random forest 0,33", "XGBoost de la malla (300 árboles)",
               "XGBoost con el λ de menor RMSE de validación cruzada"],
    "RMSE de validación cruzada (€)": [-puntuaciones_lineal.mean(), -puntuaciones_bosque.mean(),
                    -puntuaciones_xgb.mean(), -puntuaciones_lambda.mean()],
    "error típico (€)": [puntuaciones_lineal.std() / np.sqrt(5), puntuaciones_bosque.std() / np.sqrt(5),
                         puntuaciones_xgb.std() / np.sqrt(5), puntuaciones_lambda.std() / np.sqrt(5)],
})

resumen.round(0)

**Lectura de los resultados.**

- XGBoost (unos 11 000 euros) mejora al random forest de 0,33 (unos 11 200 euros) en algo más de 200 euros, y a la regresión lineal (11 956 euros) en unos 1 000.
- Con el λ de menor RMSE de validación cruzada, el RMSE de validación cruzada de XGBoost cambia unas decenas de euros como mucho, mucho menos que el error típico.

---

### Para pensar 3

¿Mejora XGBoost al random forest más que el error típico? ¿Qué harías para decidir entre los dos?

<details><summary>Respuesta</summary>

- Por poco. La diferencia, algo más de 200 euros, supera por poco el error típico de cada modelo (unos 190 y 200 euros).
- Además, en este cuaderno solo XGBoost ha pasado por una búsqueda de hiperparámetros: la fracción 0,33 del random forest está fijada de antemano. Con una fracción elegida por validación cruzada, la diferencia podría ser menor.
- Para decidir, se comparan los dos modelos ya ajustados con el mismo protocolo (las mismas particiones, la misma métrica y su error típico) y, si la diferencia no supera el error típico, se tienen en cuenta otros criterios, como el tiempo de ajuste o la facilidad de explicar el modelo.

</details>

# Ampliación (opcional). Parada temprana y otras implementaciones del boosting

## A.1 Parada temprana

- En el boosting, cada árbol nuevo añade capacidad de ajuste. Con demasiados árboles, el modelo empieza a seguir el ruido de los datos con los que se ajusta, y el error de validación sube.
- Hasta ahora hemos fijado 300 árboles.
- La **parada temprana** elige el número de árboles durante un solo ajuste, sin validación cruzada ni malla de valores, y así ahorra cálculo. Para ello, vigila el error en un conjunto de datos que se separa antes de empezar.

Cómo se procede:

1. Se aparta una parte del entrenamiento, el **conjunto apartado**.
2. Se ajustan los árboles uno a uno y, tras cada árbol, se calcula el RMSE en el conjunto apartado.
3. Cuando el RMSE no mejora durante un número de árboles fijado de antemano (`early_stopping_rounds`), el ajuste se detiene.
4. El modelo se queda con el número de árboles del menor RMSE.

- En XGBoost, el conjunto apartado se pasa a `fit` en el argumento `eval_set`.
- El `Pipeline` no aplica su preprocesador a los datos de `eval_set`, así que ajustamos la parada fuera del `Pipeline`, con los datos ya preprocesados.
- Para ello, ajustamos el preprocesador con el resto del entrenamiento (el **conjunto de ajuste**) y lo usamos para transformar los dos conjuntos.

> **Python que usamos aquí**
> - `XGBRegressor(..., early_stopping_rounds=50)` crea un XGBoost que para cuando 50 árboles seguidos no mejoran el RMSE del conjunto apartado.
> - `modelo.fit(X, y, eval_set=[(X_apartado, y_apartado)], verbose=False)` ajusta el modelo y calcula, tras cada árbol, el RMSE del conjunto apartado.
> - `modelo.evals_result()` devuelve esos RMSE, árbol a árbol.

Separamos el conjunto apartado, el 20 % del entrenamiento, y preprocesamos los dos conjuntos.

In [ ]:
# Apartamos el 20 % del entrenamiento. El resto es el conjunto de ajuste.
X_ajuste, X_apartado, y_ajuste, y_apartado = train_test_split(X_train, y_train, test_size=0.2,
                                                              random_state=SEMILLA)

# El preprocesador se ajusta con el conjunto de ajuste y transforma los dos conjuntos.
preprocesador_parada = crear_preprocesador(escalar=False)
preprocesador_parada.fit(X_ajuste)
X_ajuste_prep = preprocesador_parada.transform(X_ajuste)
X_apartado_prep = preprocesador_parada.transform(X_apartado)
print("Ajuste:", len(X_ajuste), "| apartado:", len(X_apartado))

Ajustamos la parada temprana con la tasa y la profundidad elegidas en el apartado 1.2. El modelo puede llegar a 2 000 árboles, pero se detiene si 50 árboles seguidos no mejoran el RMSE del conjunto apartado.

In [ ]:
# Parada temprana: hasta 2000 árboles. Para si 50 seguidos no mejoran el RMSE del conjunto apartado.
parada = XGBRegressor(n_estimators=2000, learning_rate=tasa, max_depth=profundidad,
                      early_stopping_rounds=50, random_state=SEMILLA)
parada.fit(X_ajuste_prep, y_ajuste, eval_set=[(X_apartado_prep, y_apartado)], verbose=False)

# best_iteration es el árbol del menor RMSE, contado desde 0: el número de árboles es best_iteration + 1.
arboles_parada = parada.best_iteration + 1
# best_score es ese RMSE del conjunto apartado.
print("Árboles elegidos:", arboles_parada, "| RMSE del conjunto apartado:", round(parada.best_score), "euros")

Dibujamos el RMSE del conjunto apartado tras cada árbol, con una línea en el número de árboles elegido.

In [ ]:
# RMSE del conjunto apartado tras cada árbol ("validation_0" es el primer conjunto de eval_set).
rmse_apartado = parada.evals_result()["validation_0"]["rmse"]
plt.plot(np.arange(1, len(rmse_apartado) + 1), rmse_apartado)

# Línea vertical en el número de árboles elegido.
plt.axvline(arboles_parada, color="grey", linestyle="--")
plt.xlabel("Número de árboles")
plt.ylabel("RMSE del conjunto apartado (euros)")
plt.show()

**Lectura de los resultados.**

- La parada elige entre 400 y 500 árboles, según el ordenador. En realidad ajusta 50 árboles más: los que necesita para comprobar que el RMSE ya no mejora.
- En el gráfico, el RMSE del conjunto apartado baja deprisa con los primeros árboles y después se aplana.
- Su mínimo es de unos 11 150 euros, y en los 50 árboles siguientes no vuelve a bajar de esa cifra.
- Ese RMSE sale de un solo ajuste y de los 1 600 trabajadores del conjunto apartado.

---

Para compararlo con los demás modelos, calculamos el RMSE de validación cruzada, con las particiones `cv5`, de un XGBoost con `arboles_parada` árboles dentro de un `Pipeline`.

In [ ]:
# XGBoost con los árboles que elige la parada temprana, en un Pipeline.
xgb_parada = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", XGBRegressor(n_estimators=arboles_parada, learning_rate=tasa, max_depth=profundidad,
                            random_state=SEMILLA)),
])

# RMSE de validación cruzada con las particiones cv5, repartidas entre los núcleos.
puntuaciones_parada = cross_val_score(xgb_parada, X_train, y_train, scoring="neg_root_mean_squared_error",
                                      cv=cv5, n_jobs=N_NUCLEOS)
rmse_parada = -puntuaciones_parada.mean()
et_parada = puntuaciones_parada.std() / np.sqrt(5)

print("XGBoost con parada temprana | RMSE de validación cruzada:", round(rmse_parada), "euros | error típico:", round(et_parada))

**Lectura de los resultados.**

- Con esos árboles, el RMSE de validación cruzada es de unos 11 000 euros, casi el mismo que con los 300 árboles de la malla. La diferencia es muy inferior al error típico (unos 190 euros).
- ¿Qué ventaja tiene la parada temprana? Elige el número de árboles con un solo ajuste, sin recurrir a `GridSearchCV` sobre una malla de valores.

---

**Un RMSE de validación cruzada algo optimista.**

- El conjunto apartado sale del entrenamiento, así que sus trabajadores también forman parte de las particiones de `cv5`.
- Es decir, hemos elegido el número de árboles mirando datos que después usamos para validar, y por eso este RMSE de validación cruzada es algo optimista.
- El procedimiento riguroso repite la parada dentro de cada partición, con un conjunto apartado que sale solo de la parte de ajuste de esa partición. Exige escribir un bucle propio y, con estos datos, cambia el RMSE de validación cruzada menos que el error típico, así que aquí nos quedamos con el procedimiento simple.
- En el apartado A.2, `HistGradientBoostingRegressor` hace la parada dentro de cada ajuste sin código adicional.

### Ejercicio 2 (opcional). La parada temprana con una tasa menor

1. Repite la parada temprana con ν = 0,03 y la misma profundidad.
2. ¿Cuántos árboles elige la parada con ν = 0,03, frente a `arboles_parada`? ¿Cambia mucho el RMSE del conjunto apartado?

In [ ]:
# TODO: copia la celda que crea el modelo parada (la del XGBRegressor con early_stopping_rounds=50) y cambia:
#   - learning_rate=tasa por learning_rate=0.03.
#   - el nombre del modelo, parada, por parada_lenta, también en la línea de fit.
#   - borra las líneas de arboles_parada y del print: arboles_parada debe seguir siendo el de la tasa de la malla.
#     Usa la línea de abajo, que muestra el resultado.

print("Tasa 0,03 | árboles elegidos:", parada_lenta.best_iteration + 1, "| RMSE del conjunto apartado:", round(parada_lenta.best_score), "euros")

Paso 2. ¿Cuántos árboles elige la parada con ν = 0,03, frente a `arboles_parada`? ¿Cambia mucho el RMSE del conjunto apartado?

*Escribe aquí tu respuesta.*

### Para pensar 4

¿Elige la parada temprana la tasa de aprendizaje?

<details><summary>Respuesta</summary>

- No. Para una tasa y una profundidad dadas, elige el número de árboles.
- La tasa y la profundidad se eligen con validación cruzada (apartado 1.2).
- Una tasa menor necesita más árboles, y el ajuste tarda más.

</details>

## A.2 HistGradientBoosting y LightGBM

XGBoost no es la única implementación del boosting de árboles. Hay otras dos muy usadas que siguen la misma idea.

- Cada árbol nuevo corrige los errores de la predicción acumulada, y su aportación se multiplica por la tasa de aprendizaje.
- Lo que cambia son algunas decisiones de diseño y los valores por defecto.

**Qué cambian.** Sobre todo, la rapidez con muchos datos.

- Para buscar los cortes de cada árbol, agrupan los valores de cada predictor en intervalos (como mucho 255, los «histogramas») y solo prueban cortes entre un intervalo y el siguiente, en lugar de probar todos los valores. XGBoost también lo hace por defecto.
- **`HistGradientBoostingRegressor`** forma parte de scikit-learn, así que no necesita otra librería ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)).
  - Incluye la parada temprana: con `early_stopping=True`, cada ajuste aparta la fracción `validation_fraction` de los datos que recibe y se detiene tras `n_iter_no_change` árboles sin mejora.
  - Dentro de `cross_validate`, cada partición hace su propia parada con su parte de ajuste. Es el procedimiento riguroso del apartado A.1, sin código adicional.
- **`LGBMRegressor`** es el modelo de la librería LightGBM con la interfaz de scikit-learn ([documentación](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html)).
  - Hace crecer cada árbol por la hoja que más reduce la pérdida y, en lugar de la profundidad, limita el número de hojas (`num_leaves`).
  - Un árbol de profundidad 4 tiene como mucho 16 hojas, así que con `num_leaves=2 ** profundidad` sus árboles tienen un tamaño parecido a los de XGBoost, aunque su forma es distinta.

Como XGBoost, las dos van en un `Pipeline` con el preprocesador sin escalar, y la validación cruzada reparte sus ajustes entre los núcleos.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# HistGradientBoosting con la tasa y la profundidad de la malla, y parada temprana dentro de cada ajuste:
# hasta 2000 árboles. Aparta el 20 % de los datos que recibe y para si 50 árboles seguidos no mejoran.
hgb = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", HistGradientBoostingRegressor(learning_rate=tasa, max_depth=profundidad, max_iter=2000,
                                             early_stopping=True, validation_fraction=0.2,
                                             n_iter_no_change=50, random_state=SEMILLA)),
])

# return_estimator=True guarda el modelo ajustado en cada partición, con su número de árboles (n_iter_).
resultados_hgb = cross_validate(hgb, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv5,
                                return_estimator=True, n_jobs=N_NUCLEOS)
print("Árboles elegidos en cada partición:", [ajuste["modelo"].n_iter_ for ajuste in resultados_hgb["estimator"]])

Ajustamos LightGBM con la tasa y los árboles de la parada temprana, y reunimos los tres boostings en una tabla.

In [ ]:
from lightgbm import LGBMRegressor

# LightGBM con la tasa y los árboles de la parada temprana, y hasta 2 ** profundidad hojas por árbol.
# n_jobs=1: un núcleo por ajuste (sin él, LightGBM es mucho más lento con la validación cruzada repartida).
# verbose=-1 quita los mensajes de LightGBM durante el ajuste.
lgbm = Pipeline([
    ("preprocesado", crear_preprocesador(escalar=False)),
    ("modelo", LGBMRegressor(n_estimators=arboles_parada, learning_rate=tasa, num_leaves=2 ** profundidad,
                             random_state=SEMILLA, n_jobs=1, verbose=-1)),
])

puntuaciones_lgbm = cross_val_score(lgbm, X_train, y_train, scoring="neg_root_mean_squared_error",
                                    cv=cv5, n_jobs=N_NUCLEOS)

# Los tres boostings, con las particiones cv5.
otros_boostings = pd.DataFrame({
    "modelo": ["XGBoost con parada temprana", "HistGradientBoosting", "LightGBM"],
    "RMSE de validación cruzada (€)": [rmse_parada, -resultados_hgb["test_score"].mean(), -puntuaciones_lgbm.mean()],
    "error típico (€)": [et_parada, resultados_hgb["test_score"].std() / np.sqrt(5),
                         puntuaciones_lgbm.std() / np.sqrt(5)],
})

otros_boostings.round(0)

**Lectura de los resultados.**

- HistGradientBoosting elige en cada partición su propio número de árboles, y en estos datos son menos que `arboles_parada`.
- Su RMSE de validación cruzada y el de LightGBM quedan algo por encima del de XGBoost, a una distancia del orden del error típico o menor.
- Hay que tener en cuenta que cada implementación tiene sus propios valores por defecto. Por ejemplo, HistGradientBoosting y LightGBM exigen al menos 20 trabajadores por hoja, y XGBoost admite hojas de un solo trabajador.
- La tabla compara estos tres modelos, cada uno con sus valores por defecto en los hiperparámetros que no hemos fijado.

---

### Para pensar 5

¿Por qué el RMSE de validación cruzada de HistGradientBoosting no tiene el optimismo del de la parada temprana del apartado A.1?

<details><summary>Respuesta</summary>

- En cada partición de `cv5`, HistGradientBoosting aparta su 20 % de la parte de ajuste de esa partición y elige ahí el número de árboles.
- Los trabajadores de validación de la partición no intervienen en la elección, como en la validación cruzada de cualquier otro modelo.
- En el apartado A.1, el número de árboles se eligió una sola vez, con un conjunto apartado cuyos trabajadores están después en las particiones de validación.

</details>

**Para experimentar.**

- Quita `max_depth` de `HistGradientBoostingRegressor`: sus árboles quedan limitados por `max_leaf_nodes`, 31 hojas por defecto.
- Deja LightGBM con sus valores por defecto: `LGBMRegressor(random_state=SEMILLA, n_jobs=1, verbose=-1)`.
- Usa `validation_fraction=0.1` y mira cuántos árboles elige HistGradientBoosting en cada partición.